In [1]:
pip install torch

In [2]:
pip install dice-ml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.8 MB/s eta 0:00:00


In [3]:
# %% Imports
from torch.utils.data import DataLoader
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score

import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

In [33]:
# %% Custom DataLoader
class CustomDataLoader:
    def __init__(self, filepath):
        self.filepath = filepath
        self.data = None

    def load_dataset(self):
        self.data = pd.read_csv(self.filepath)

    def preprocess_data(self):
        # Implement your preprocessing here
        self.data.dropna(inplace=True)
        # Drop 'id' column as it's not a feature
        if 'id' in self.data.columns:
            self.data.drop('id', axis=1, inplace=True)
        self.data = pd.get_dummies(self.data)
        # Convert any boolean or uint8 columns (from get_dummies) to int (0 or 1)
        for col in self.data.select_dtypes(include=['bool', 'uint8']).columns:
            self.data[col] = self.data[col].astype(int)

    def get_data_split(self, test_size=0.2, random_state=42):
        X = self.data.drop('stroke', axis=1)
        y = self.data['stroke']
        return train_test_split(X, y, test_size=test_size, random_state=random_state)

    def oversample(self, X_train, y_train):
        smote = SMOTE(random_state=42)
        X_res, y_res = smote.fit_resample(X_train, y_train)
        return X_res, y_res

# %% Load and preprocess data
data_loader = CustomDataLoader('/content/healthcare-dataset-stroke-data.csv')
data_loader.load_dataset()
data_loader.preprocess_data()

In [34]:
# Split the data for evaluation
X_train, X_test, y_train, y_test = data_loader.get_data_split()

# Oversample the train data
X_train, y_train = data_loader.oversample(X_train, y_train)


y_test = y_test.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_train = X_train.reset_index(drop=True)

In [35]:
# %% Fit blackbox model
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(f"F1 Score {f1_score(y_test, y_pred, average='macro')}")
print(f"Accuracy {accuracy_score(y_test, y_pred)}")

F1 Score 0.4842436974789916
Accuracy 0.9389002036659878


In [36]:
# Convert y_test and y_pred to pandas Series
y_test_series = pd.Series(y_test)
y_pred_series = pd.Series(y_pred)

# Get indices where y_test and y_pred are 1
test_indices = y_test_series[y_test_series == 1].index.tolist()
pred_indices = y_pred_series[y_pred_series == 1].index.tolist()

print("Test indices:", test_indices)
print("Prediction indices:", pred_indices)

Test indices: [11, 30, 35, 62, 73, 110, 113, 122, 143, 166, 167, 198, 229, 238, 274, 277, 299, 312, 327, 336, 362, 388, 426, 434, 475, 488, 499, 538, 573, 582, 592, 598, 613, 682, 685, 734, 752, 795, 804, 807, 809, 832, 851, 873, 878, 903, 910, 917, 928, 944, 964, 965, 978]
Prediction indices: [199, 255, 288, 390, 614, 679, 785]


In [37]:
# %% Create diverse counterfactual explanations
import dice_ml

# Dataset
# Get all categorical columns
all_cols = data_loader.data.columns.tolist()
continuous_features = ['age',
                                              'avg_glucose_level',
                                              'bmi']
outcome_name = 'stroke'
categorical_features = [col for col in all_cols if col not in continuous_features + [outcome_name]] # 'id' is removed earlier

data_dice = dice_ml.Data(dataframe=data_loader.data,
                         # For perturbation strategy
                         continuous_features=continuous_features,
                         categorical_features=categorical_features, # Explicitly define categorical features
                         outcome_name=outcome_name)

In [38]:
# Model
rf_dice = dice_ml.Model(model=rf,
                        # There exist backends for tf, torch, ...
                        backend="sklearn")
explainer = dice_ml.Dice(data_dice,
                         rf_dice,
                         # Random sampling, genetic algorithm, kd-tree,...
                         method="random")

In [39]:
import numpy as np

# %% Create explanation
# Generate CF based on the blackbox model
input_datapoint = X_test[10:11].copy()

# Convert boolean columns in input_datapoint to int (0 or 1)
for col in input_datapoint.select_dtypes(include='bool').columns:
    input_datapoint[col] = input_datapoint[col].astype(int)

cf = explainer.generate_counterfactuals(input_datapoint,
                                  total_CFs=3,
                                  desired_class="opposite")

  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is depr

In [40]:
print(X_test[0:1])

    age  hypertension  heart_disease  avg_glucose_level   bmi  gender_Female  \
0  80.0             0              1             125.32  32.9              0   

   gender_Male  gender_Other  ever_married_No  ever_married_Yes  ...  \
0            1             0                0                 1  ...   

   work_type_Never_worked  work_type_Private  work_type_Self-employed  \
0                       0                  1                        0   

   work_type_children  Residence_type_Rural  Residence_type_Urban  \
0                   0                     1                     0   

   smoking_status_Unknown  smoking_status_formerly smoked  \
0                       1                               0   

   smoking_status_never smoked  smoking_status_smokes  
0                            0                      0  

[1 rows x 21 columns]


In [41]:
# Visualize it
# cf.visualize_as_dataframe(show_only_changes=False)

cf.visualize_as_dataframe(show_only_changes=True)

Query instance (original outcome : 0)


,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,ever_married_Yes,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,73.0,0,0,87.559998,24.1,1,0,0,0,1,...,0,1,0,0,1,0,0,1,0,0



Diverse Counterfactual set (new outcome: 1)


,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,ever_married_Yes,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,-,-,-,-,-,-,-,1.0,-,-,...,-,0.0,-,-,-,-,-,-,-,1.0
1,-,-,-,-,-,-,-,-,-,-,...,-,-,-,-,0.0,-,-,-,-,1.0
2,-,-,-,-,-,-,-,-,-,-,...,-,-,-,-,-,-,-,0.0,-,1.0


In [28]:
# Get indices where age is above 70
indices_above_70 = X_test[X_test['age'] > 70].index.tolist()

print("Indices of people whose age is above 70:", indices_above_70)

Indices of people whose age is above 70: [0, 10, 22, 39, 56, 62, 63, 76, 79, 88, 106, 110, 113, 131, 139, 142, 143, 147, 183, 195, 202, 205, 206, 208, 216, 222, 235, 240, 247, 255, 258, 265, 267, 272, 274, 276, 277, 288, 304, 311, 312, 320, 324, 327, 336, 347, 349, 357, 362, 370, 376, 388, 389, 417, 419, 426, 432, 434, 443, 462, 465, 466, 468, 475, 480, 486, 492, 495, 501, 505, 542, 556, 575, 582, 596, 598, 601, 605, 611, 618, 634, 635, 640, 647, 651, 652, 658, 660, 668, 675, 679, 683, 685, 702, 709, 710, 722, 724, 732, 734, 749, 753, 764, 782, 785, 787, 790, 792, 799, 803, 807, 809, 824, 848, 851, 878, 880, 892, 895, 903, 907, 926, 938, 942, 943, 944, 949, 952, 960, 961, 971, 978, 980]


In [42]:
# %% Create feasible (conditional) Counterfactuals
features_to_vary=['avg_glucose_level',
                  'bmi',
                  'smoking_status_smokes',
                  'age',
                  'hypertension']
permitted_range={'avg_glucose_level':[40,300],
                'bmi':[15, 45],
                'age':[18, 90]}

i = 139

input_datapoint2 = X_test[i:i+1].copy() # Ensure it's a copy

# Convert boolean columns in input_datapoint2 to int (0 or 1)
for col in input_datapoint2.select_dtypes(include='bool').columns:
    input_datapoint2[col] = input_datapoint2[col].astype(int)

print("Label of test data: ", y_test[i])
print(input_datapoint2.to_string(index=False))

# Now generating explanations using the new feature weights

cf = explainer.generate_counterfactuals(input_datapoint2,
                                  total_CFs=10,
                                  desired_class="opposite",
                                  permitted_range=permitted_range,
                                  features_to_vary=features_to_vary)
# Visualize it
cf.visualize_as_dataframe(show_only_changes=True)

Label of test data:  0
 age  hypertension  heart_disease  avg_glucose_level  bmi  gender_Female  gender_Male  gender_Other  ever_married_No  ever_married_Yes  work_type_Govt_job  work_type_Never_worked  work_type_Private  work_type_Self-employed  work_type_children  Residence_type_Rural  Residence_type_Urban  smoking_status_Unknown  smoking_status_formerly smoked  smoking_status_never smoked  smoking_status_smokes
81.0             0              0              125.2 40.0              1            0             0                0                 1                   0                       0                  0                        1                   0                     0                     1                       0                               0                            1                      0


  0%|          | 0/1 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
/usr/local/lib/python3.12/dist-packages/dice_ml/explainer_interfaces/dice_random.py:116: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '1' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  candidate_cfs.at[k, selected_features[k][0]] = random_instances.at[k, selected_features[k][0]]
100%|██████████| 1/1 [00:00<00:00,  1.26it/s]

Query instance (original outcome : 0)


,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,ever_married_Yes,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,81.0,0,0,125.199997,40.0,1,0,0,0,1,...,0,1,0,0,1,0,0,1,0,0



Diverse Counterfactual set (new outcome: 1)


,age,hypertension,heart_disease,avg_glucose_level,bmi,gender_Female,gender_Male,gender_Other,ever_married_No,ever_married_Yes,...,work_type_Private,work_type_Self-employed,work_type_children,Residence_type_Rural,Residence_type_Urban,smoking_status_Unknown,smoking_status_formerly smoked,smoking_status_never smoked,smoking_status_smokes,stroke
0,-,-,-,72.64,32.5,-,-,-,-,-,...,-,-,-,-,-,-,-,-,-,1.0


In [44]:
print(y_test[i])

0
